In [1]:
# --- Load Libraries ---
import vireoSNP
from vireoSNP import BinomMixtureVB
import numpy as np
from scipy.io import mmread
import pandas as pd

print("vireoSNP version:", vireoSNP.__version__)

vireoSNP version: 0.5.9


In [12]:
# --- Snakemake Integration ---
# Inputs
# in_passed_ad = snakemake.input.passed_ad
in_passed_ad = "../../../results/standalone/mquad/TNBC1/passed_ad.mtx"
# in_passed_dp = snakemake.input.passed_dp
in_passed_dp = "../../../results/standalone/mquad/TNBC1/passed_dp.mtx"
# in_barcodes = snakemake.input.barcodes
in_barcodes = "../../../../data/MQuad_test/TNBC1/cellSNP.samples.tsv"
# in_variant_names = snakemake.input.variant_names
in_variant_names = "../../../results/standalone/mquad/TNBC1/passed_variant_names.txt"

# Outputs
# out_clones_df = snakemake.output.clones_df
out_clones_df = "../../../results/standalone/vireo/TNBC1/barcodes_donor_ids.csv"
# out_variant_df = snakemake.output.variant_df
out_variant_df = "../../../results/standalone/vireo/TNBC1/variant_donor.csv"

# Parameters
# n_clones_to_test = snakemake.params.n_clones_list
n_clones_to_test = [2, 3, 4, 5]

In [7]:
# --- 1. Load Data ---
AD = mmread(in_passed_ad).tocsc()
DP = mmread(in_passed_dp).tocsc()

with open(in_barcodes, 'r') as f:
    sample_id = f.read().splitlines()

with open(in_variant_names, 'r') as f:
    variant_names = f.read().splitlines()

In [8]:
# --- 2. Initialize Master DataFrames ---
clones_df = pd.DataFrame({'sample_id': sample_id})
variant_df = pd.DataFrame(index=variant_names)
variant_df.index.name = "Variant_Name"

In [10]:
clones_df

,sample_id
0,AAACCTGCACCTTGTC-1
1,AAACGGGAGTCCTCCT-1
2,AAACGGGTCCAGAGGA-1
3,AAAGATGCAGTTTACG-1
4,AAAGCAACAGGAATGC-1
...,...
1092,TTTATGCTCCTCATTA-1
1093,TTTATGCTCTGTTGAG-1
1094,TTTCCTCTCGGAAACG-1
1095,TTTGCGCCAATCACAC-1


In [9]:
variant_df

""
Variant_Name
4076C>T
310T>C


In [11]:
# --- 3. Iterate over the range of clones ---
for n_clones in n_clones_to_test:
    print(f"Running vireoSNP for N={n_clones} clones...")
    
    # Initialize and fit the model
    _model = BinomMixtureVB(n_var=AD.shape[0], n_cell=AD.shape[1], n_donor=n_clones)
    _model.fit(AD, DP, min_iter=30, n_init=300)
    
    # --- Process Cells ---
    prob_matrix = _model.ID_prob
    clone_id = np.argmax(prob_matrix, axis=1)
    conf = np.max(prob_matrix, axis=1) >= 0.8
    
    clones_df[f'clone_id_N{n_clones}'] = clone_id
    clones_df[f'confident_N{n_clones}'] = conf
    
    for i in range(n_clones):
        clones_df[f'prob_N{n_clones}_clone_{i}'] = prob_matrix[:, i]
        
    # --- Process Variants ---
    beta_mu_matrix = _model.beta_mu
    for i in range(n_clones):
        variant_df[f'N{n_clones}_Clone_{i}_VAF'] = beta_mu_matrix[:, i]

Running vireoSNP for N=2 clones...
Running vireoSNP for N=3 clones...
Running vireoSNP for N=4 clones...
Running vireoSNP for N=5 clones...


In [14]:
clones_df

,sample_id,clone_id_N2,confident_N2,prob_N2_clone_0,prob_N2_clone_1,clone_id_N3,confident_N3,prob_N3_clone_0,prob_N3_clone_1,prob_N3_clone_2,...,prob_N4_clone_1,prob_N4_clone_2,prob_N4_clone_3,clone_id_N5,confident_N5,prob_N5_clone_0,prob_N5_clone_1,prob_N5_clone_2,prob_N5_clone_3,prob_N5_clone_4
0,AAACCTGCACCTTGTC-1,0,True,0.999985,1.534203e-05,1,False,1.249374e-06,0.743342,0.256657,...,0.421505,4.788971e-07,0.101113,3,False,0.246782,0.041196,0.355643,0.356379,1.925645e-07
1,AAACGGGAGTCCTCCT-1,0,True,0.984272,1.572839e-02,1,False,4.013167e-03,0.500941,0.495045,...,0.335013,2.179691e-03,0.329110,2,False,0.242608,0.251924,0.252086,0.252086,1.295744e-03
2,AAACGGGTCCAGAGGA-1,0,True,0.999431,5.690122e-04,1,False,8.618097e-05,0.670162,0.329752,...,0.404452,3.989793e-05,0.156093,3,False,0.259144,0.078624,0.330878,0.331334,1.982255e-05
3,AAAGATGCAGTTTACG-1,0,True,0.873093,1.269068e-01,1,False,5.122059e-02,0.516374,0.432405,...,0.344660,3.189733e-02,0.271585,3,False,0.252734,0.187676,0.268806,0.268899,2.188519e-02
4,AAAGCAACAGGAATGC-1,0,True,0.999994,6.055467e-06,2,False,3.885732e-07,0.327366,0.672634,...,0.267231,1.409268e-07,0.508093,1,False,0.206692,0.440470,0.176650,0.176188,5.318791e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092,TTTATGCTCCTCATTA-1,0,True,0.998354,1.645734e-03,1,True,2.721690e-04,0.826970,0.172758,...,0.427324,1.306983e-04,0.052461,3,False,0.237260,0.014744,0.373384,0.374545,6.725864e-05
1093,TTTATGCTCTGTTGAG-1,0,True,0.996168,3.832426e-03,1,False,7.934958e-04,0.588528,0.410679,...,0.376225,4.053067e-04,0.231900,3,False,0.259056,0.145085,0.297714,0.297919,2.252578e-04
1094,TTTCCTCTCGGAAACG-1,0,True,0.999607,3.926604e-04,2,False,5.536236e-05,0.404136,0.595809,...,0.322056,2.555477e-05,0.393557,1,False,0.257363,0.281100,0.230984,0.230540,1.279714e-05
1095,TTTGCGCCAATCACAC-1,0,True,1.000000,1.377693e-11,1,False,1.137287e-13,0.642774,0.357226,...,0.400776,2.189894e-14,0.180537,3,False,0.229109,0.111769,0.329391,0.329730,4.098180e-15


In [15]:
variant_df

,N2_Clone_0_VAF,N2_Clone_1_VAF,N3_Clone_0_VAF,N3_Clone_1_VAF,N3_Clone_2_VAF,N4_Clone_0_VAF,N4_Clone_1_VAF,N4_Clone_2_VAF,N4_Clone_3_VAF,N5_Clone_0_VAF,N5_Clone_1_VAF,N5_Clone_2_VAF,N5_Clone_3_VAF,N5_Clone_4_VAF
Variant_Name,,,,,,,,,,,,,,
4076C>T,0.000429,0.216256,0.247318,0.000281,0.000977,0.000529,0.000298,0.256480,0.001343,0.002451,0.000238,0.000200,0.000200,0.266637
310T>C,0.156036,0.140607,0.128666,0.093969,0.237024,0.099198,0.119254,0.128441,0.300067,0.143133,0.374695,0.104945,0.104636,0.129409


In [ ]:
# --- 4. Save Final Master Tables ---
clones_df.to_csv(out_clones_df, index=False)
variant_df.to_csv(out_variant_df)

print(f"\nSuccessfully saved {out_clones_df} and {out_variant_df}")